In [4]:
import pandas as pd

# ===== Load file =====
file_path = "micro_climate_rh_t_et0.xlsx"
df = pd.read_excel(file_path)

# ===== Keep G numeric =====
g_col = 'G,MJ/m2/h(day/night)'
df[g_col] = pd.to_numeric(df[g_col], errors='coerce')
df = df.dropna(subset=[g_col]).reset_index(drop=True)

# ===== Parse only the first timestamp safely =====
first_raw = str(df.loc[0, 'timestamp_dayfirst']).strip()

# try common formats for the first row
start_ts = pd.to_datetime(first_raw, format='%d/%m/%Y %H:%M:%S', errors='coerce')
if pd.isna(start_ts):
    start_ts = pd.to_datetime(first_raw, format='%d/%m/%Y %H:%M', errors='coerce')
if pd.isna(start_ts):
    start_ts = pd.to_datetime(first_raw, format='%d/%m/%y %H:%M:%S', errors='coerce')
if pd.isna(start_ts):
    start_ts = pd.to_datetime(first_raw, format='%d/%m/%y %H:%M', errors='coerce')

if pd.isna(start_ts):
    raise ValueError(f"Could not parse first timestamp: {first_raw}")

# ===== Rebuild full timestamp column as strict 10-minute sequence =====
df['timestamp_fixed'] = pd.date_range(
    start=start_ts,
    periods=len(df),
    freq='10min'
)

# ===== Daily sum of G =====
daily_G = (
    df.set_index('timestamp_fixed')[g_col]
      .resample('D')
      .sum()
      .reset_index()
)

daily_G.columns = ['date', 'daily_G_sum']
daily_G['date'] = daily_G['date'].dt.strftime('%d/%m/%Y')

print(daily_G)

# ===== Save =====
output_file = "daily_G_sum.xlsx"
daily_G.to_excel(output_file, index=False)

print(f"\nSaved to: {output_file}")

           date  daily_G_sum
0    29/05/2025     1.324401
1    30/05/2025     1.396559
2    31/05/2025     1.436143
3    01/06/2025     1.483181
4    02/06/2025     1.456430
..          ...          ...
111  17/09/2025     0.831087
112  18/09/2025     0.841865
113  19/09/2025     0.772624
114  20/09/2025     0.896353
115  21/09/2025     0.853155

[116 rows x 2 columns]

Saved to: daily_G_sum.xlsx
